# Geometric Knowledge Network Demo

This notebook keeps orchestration light and delegates implementation to the package in `src/`.

In [ ]:
from pathlib import Path
import sys

sys.path.append(str(Path.cwd().parent / 'src'))

from geometric_knowledge_network.config import GKNConfig
from geometric_knowledge_network.extraction import ConceptExtractor
from geometric_knowledge_network.graph_builder import KnowledgeNetworkBuilder
from geometric_knowledge_network.hybrid_retriever import HybridRetriever
from geometric_knowledge_network.ingest import DocumentIngestor
from geometric_knowledge_network.vector_store import SimpleVectorStore
from geometric_knowledge_network.visualization import draw_subgraph

In [ ]:
config = GKNConfig()
ingestor = DocumentIngestor()
documents = ingestor.load_text_documents(config.sample_docs_dir)
chunks = ingestor.chunk_documents(documents, config.chunk_size, config.chunk_overlap)

len(documents), len(chunks)

In [ ]:
vector_store = SimpleVectorStore()
vector_store.build(chunks)

query = 'What evidence is required for validation approval?'
baseline_results = vector_store.search(query, top_k=config.top_k)
[(r.chunk_id, round(r.score, 3)) for r in baseline_results]

In [ ]:
extractor = ConceptExtractor(config.concept_keywords)
graph = KnowledgeNetworkBuilder(extractor).build(documents, chunks)
graph.number_of_nodes(), graph.number_of_edges()

In [ ]:
hybrid = HybridRetriever(vector_store, graph)
hybrid_results = hybrid.search(query, top_k=config.top_k, graph_hops=config.graph_hops)
[(r.chunk_id, round(r.vector_score, 3), round(r.graph_bonus, 3), round(r.final_score, 3)) for r in hybrid_results]

In [ ]:
for result in hybrid_results:
    print('=' * 80)
    print(result.chunk_id, '| final_score=', round(result.final_score, 3))
    print(result.text[:500])

In [ ]:
draw_subgraph(graph, hybrid_results[0].chunk_id, radius=2)